# 05 — Kalle LLMs: Ollama, Groq og OpenAI-kompatible APIer

**Fase:** 2 — Kjerne AI | **Tid:** 2 timer | **Krav:** Notatbok 04

**Hva du bygger:** En funksjonell LLM-klient som snakker med en lokal gratismodell, med streaming og tool calling.

---

## Gratis alternativer

| Alternativ | Gratis? | Krever nett? | Notater |
|------------|---------|-------------|--------|
| **Ollama** (anbefalt) | ✅ 100% | ❌ Nei — lokalt | Kjør Llama, Mistral, Gemma lokalt |
| **Groq** | ✅ Gratiskvote | ✅ Ja | Lynrask inferens, enkel API |
| **Google Gemini** | ✅ Gratiskvote | ✅ Ja | Gemini 1.5 Flash er gratis |
| **Hugging Face** | ✅ Gratiskvote | ✅ Ja | Mange modeller |
| OpenAI / Anthropic | ❌ Betalt | ✅ Ja | Beste modeller, koster penger |

**Vi bruker Ollama** — ingen konto, ingen kredittkort, ingen grenser.

---

## Steg 1: Installer Ollama

```bash
# Mac/Linux:
curl -fsSL https://ollama.com/install.sh | sh

# Last ned en modell (velg én):
ollama pull llama3.2        # 2 GB — rask og god
ollama pull mistral         # 4 GB — veldig god
ollama pull gemma2:2b       # 1.6 GB — minste/raskeste

# Start serveren (kjør i terminal):
ollama serve
```

Ollama kjører på `http://localhost:11434` og eksponerer et OpenAI-kompatibelt API.

In [ ]:
%pip install -q openai requests

In [ ]:
import requests

# Sjekk at Ollama kjører og se hvilke modeller du har
try:
    resp = requests.get("http://localhost:11434/api/tags", timeout=3)
    modeller = [m["name"] for m in resp.json().get("models", [])]
    print("Ollama kjører! Tilgjengelige modeller:")
    for m in modeller:
        print(f"  - {m}")
    MODELL = modeller[0] if modeller else "llama3.2"
    print(f"\nBruk: {MODELL}")
except Exception as e:
    print(f"Ollama ikke funnet: {e}")
    print("Kjør: ollama serve")

---

## Del 1: Grunnleggende chat-API

Alle store LLM-leverandører (OpenAI, Groq, Anthropic, Ollama) bruker samme mønster:
en liste med **meldinger** der hver har en **rolle** (`system`, `user`, `assistant`).

In [ ]:
from openai import OpenAI

# Ollama er OpenAI-kompatibelt — bytt bare base_url
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Kan være hva som helst
)

# De tre rollene:
# system   = instruks til modellen (hvem er den, hvordan skal den svare)
# user     = brukerens spørsmål
# assistant = modellens svar (brukes i flerturas samtaler)

svar = client.chat.completions.create(
    model=MODELL,
    messages=[
        {"role": "system",  "content": "Du er en kortfattet pensjonsrådgiver. Svar på norsk i 2-3 setninger."},
        {"role": "user",    "content": "Hva er AFP?"},
    ],
    temperature=0.3,
)

print(svar.choices[0].message.content)
print(f"\n[Brukt: {svar.usage.prompt_tokens} input + {svar.usage.completion_tokens} output tokens]")

---

## Del 2: Samtalehistorikk

LLMs har ikke hukommelse mellom kall — du sender historikken selv hver gang.

In [ ]:
def chat(historikk: list, spørsmål: str, system: str = "Svar kort på norsk.") -> str:
    """Enkel chatfunksjon med samtalehistorikk."""
    meldinger = [{"role": "system", "content": system}] + historikk
    meldinger.append({"role": "user", "content": spørsmål})
    
    svar = client.chat.completions.create(
        model=MODELL,
        messages=meldinger,
        temperature=0.3,
    )
    
    assistant_svar = svar.choices[0].message.content
    # Legg til ny tur i historikken
    historikk.append({"role": "user",      "content": spørsmål})
    historikk.append({"role": "assistant", "content": assistant_svar})
    return assistant_svar

system = "Du er en pensjonsrådgiver hos SPK. Svar konsist på norsk."
historikk = []

svar1 = chat(historikk, "Hva er AFP?", system)
print(f"Bruker: Hva er AFP?\nModell: {svar1}\n")

svar2 = chat(historikk, "Fra hvilken alder kan jeg ta det ut?", system)
print(f"Bruker: Fra hvilken alder kan jeg ta det ut?\nModell: {svar2}")
# Modellen husker at vi snakker om AFP fra forrige tur

---

## Del 3: Streaming

Streaming viser svar ord-for-ord mens modellen genererer — viktig for god UX.

In [ ]:
import sys

stream = client.chat.completions.create(
    model=MODELL,
    messages=[
        {"role": "system", "content": "Du er en pensjonsveileder. Svar på norsk."},
        {"role": "user",   "content": "Forklar forskjellen mellom alderspensjon og AFP."},
    ],
    temperature=0.3,
    stream=True,  # Aktiver streaming
)

print("Streaming svar:")
for chunk in stream:
    tekst = chunk.choices[0].delta.content
    if tekst:
        print(tekst, end="", flush=True)  # Skriv ut uten linjeskift
print()  # Avslutt med linjeskift

---

## Del 4: Prompt engineering

Kvaliteten på svaret avhenger mye av hvordan du stiller spørsmålet.

In [ ]:
def spør(system: str, spørsmål: str) -> str:
    r = client.chat.completions.create(
        model=MODELL,
        messages=[{"role": "system", "content": system},
                  {"role": "user",   "content": spørsmål}],
        temperature=0.0,
    )
    return r.choices[0].message.content

# Teknikk 1: Vag prompt vs. strukturert prompt
vag    = spør("Du er en assistent.", "AFP")
presis = spør(
    "Du er en pensjonsrådgiver. Svar på norsk. Bruk maks 3 kulepunkter.",
    "Hva er AFP, og hvem har rett til det? Svar kun om norsk AFP."
)

print("=== VAG PROMPT ===")
print(vag[:300])
print("\n=== STRUKTURERT PROMPT ===")
print(presis)

In [ ]:
# Teknikk 2: Few-shot — vis eksempler på ønsket format
few_shot_system = """Du klassifiserer pensjonsspørsmål i kategorier.

Eksempler:
Q: "Når kan jeg ta ut AFP?" → Kategori: AFP
Q: "Hva skjer med pensjonen om jeg dør?" → Kategori: Etterlattepensjon  
Q: "Jeg er syk og kan ikke jobbe" → Kategori: Uførepensjon

Svar KUN med: Kategori: <navn>"""

spørsmål_liste = [
    "Fra hvilken alder kan jeg pensjonere meg?",
    "Min ektefelle er død, hva har jeg krav på?",
    "Jeg har 40% stilling, hvordan beregnes pensjonen?",
]

for q in spørsmål_liste:
    kategori = spør(few_shot_system, q)
    print(f"Q: {q}")
    print(f"A: {kategori}\n")

---

## Del 5: Tool calling (funksjonskall)

**Tool calling** lar modellen si: "Jeg trenger å slå opp noe" — og du kjører koden.

Dette er fundamentet for AI-agenter (notatbok 10).

In [ ]:
import json

# Definer et verktøy modellen kan "bruke"
verktøy = [{
    "type": "function",
    "function": {
        "name": "hent_pensjonssats",
        "description": "Hent gjeldende pensjonssats for en gitt pensjonstype",
        "parameters": {
            "type": "object",
            "properties": {
                "pensjonstype": {
                    "type": "string",
                    "enum": ["alderspensjon", "uførepensjon", "AFP"],
                    "description": "Type pensjon"
                }
            },
            "required": ["pensjonstype"]
        }
    }
}]

# Simulert database (i praksis ville dette treffe en ekte API eller DB)
SATSER = {
    "alderspensjon": "66% av sluttlønn, justert etter opptjening",
    "uførepensjon":  "66% av pensjonsgivende inntekt",
    "AFP":           "Livsvarig påslag, avhenger av opptjeningstid",
}

def hent_pensjonssats(pensjonstype: str) -> str:
    return SATSER.get(pensjonstype, "Ukjent pensjonstype")

# Runde 1: Modellen bestemmer seg for å bruke verktøyet
meldinger = [
    {"role": "system", "content": "Du er en pensjonsrådgiver. Bruk tilgjengelige verktøy for å hente nøyaktige satser."},
    {"role": "user",   "content": "Hva er satsen for uførepensjon?"},
]

svar = client.chat.completions.create(
    model=MODELL, messages=meldinger,
    tools=verktøy, tool_choice="auto",
)

melding = svar.choices[0].message

if melding.tool_calls:
    kall = melding.tool_calls[0]
    args = json.loads(kall.function.arguments)
    print(f"Modellen vil bruke: {kall.function.name}({args})")
    
    # Kjør verktøyet
    resultat = hent_pensjonssats(**args)
    print(f"Verktøyresultat: {resultat}")
    
    # Runde 2: Send resultatet tilbake og få endelig svar
    meldinger += [
        melding,
        {"role": "tool", "tool_call_id": kall.id, "content": resultat}
    ]
    
    endelig = client.chat.completions.create(model=MODELL, messages=meldinger)
    print(f"\nEndelig svar: {endelig.choices[0].message.content}")
else:
    print(f"Direkte svar: {melding.content}")

---

## Bruke Groq i stedet (gratis cloud-alternativ)

```python
# Lag gratis konto på groq.com → API Keys
# Bytt kun base_url og api_key:

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="gsk_...",  # Din Groq-nøkkel
)
MODELL = "llama-3.3-70b-versatile"  # Gratis og svært god

# Resten av koden er identisk!
```

---

## Oppsummering

- Chat-API: liste av meldinger med roller (`system` / `user` / `assistant`)
- Historikk sendes manuelt — LLMs er statsløse
- Streaming: `stream=True` for token-for-token output
- Prompt engineering: strukturerte prompts + few-shot gir mye bedre svar
- Tool calling: modellen ber om data, du kjører koden — grunnlaget for agenter
- Ollama og Groq er gratis og OpenAI-kompatible

---

## Hva er neste steg?

**Neste: `06_embeddings.ipynb`** — Tekst om til tall. Du lærer hva embedding-vektorer er, lager dem lokalt med `sentence-transformers` (gratis), og måler likhet mellom setninger.